# 动手练习：脑连接组：结构与功能连接

来自《生物医学网络科学》（Biomedical Network Science）。

每个任务先运行“准备”格，再在“你的代码”格中完成 TODO，最后运行“核对”格：打印 ✓ 表示通过，否则会给出提示。数据文件放在与本笔记本同级的 `data/` 文件夹中。

## 任务 1：从“BOLD 时间序列”到功能连接矩阵

本练习前 5 个任务使用 numpy 生成的**模拟数据**，只用于演示方法，不代表真实人脑。模拟设定：40 个“脑区”分属 4 个“功能系统”（每个 10 个脑区），200 个时间点；每个脑区的信号 = 所属系统的共同波动 + 微弱的全脑共同波动 + 独立噪声；脑区 9 额外接收系统 1 的信号，模拟一个跨系统的“连接枢纽”。

计算：

- `R`：40×40 的皮尔逊相关矩阵（`np.corrcoef` 默认每一行是一个变量）；
- `Z`：Fisher $z$ 变换后的矩阵（先把对角线设为 0，避免 $\operatorname{artanh}(1)=\infty$）；
- `r_within`、`r_between`：同一系统内（不含对角线）与不同系统之间的平均 $r$。

期望看到：系统内平均相关明显高于系统间；热图呈 4 个对角块。系统间平均相关略大于 0，来自模拟中的全脑共同波动（类比真实数据中的全局信号）。

> **提示：** `same = labels[:, None] == labels[None, :]` 给出“同一系统”的布尔矩阵。

In [ ]:
# 准备（直接运行）
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt


def simulate_bold(n_mod=4, per=10, T=200, seed=0):
    """模拟数据，仅用于演示方法。返回 (时间序列 N×T, 系统标签)。"""
    rng = np.random.default_rng(seed)
    labels = np.repeat(np.arange(n_mod), per)
    sys_sig = rng.standard_normal((n_mod, T))
    glob = rng.standard_normal(T)
    load = rng.uniform(0.4, 1.0, size=n_mod * per)     # 各脑区对本系统的载荷不同
    X = load[:, None] * sys_sig[labels] + 0.3 * glob + rng.standard_normal((n_mod * per, T))
    X[9] += 0.8 * sys_sig[1]                            # 脑区 9：跨系统连接枢纽
    return X, labels


X, labels = simulate_bold()
N = X.shape[0]
same = labels[:, None] == labels[None, :]
off = ~np.eye(N, dtype=bool)

In [ ]:
R = None          # TODO: 相关矩阵
Z = None          # TODO: Fisher z（对角线先置 0）
r_within = None   # TODO
r_between = None  # TODO

In [ ]:
# 核对（不用修改）
assert R is not None and Z is not None, "R 或 Z 还没有赋值"
assert R.shape == (40, 40), "R 应为 40×40；检查 np.corrcoef 的输入是否是“脑区 × 时间”"
assert np.allclose(R, R.T) and np.allclose(np.diag(R), 1), "相关矩阵应对称、对角线为 1"
assert np.all(np.isfinite(Z)), "Z 中有无穷大：做 arctanh 之前要把对角线设为 0"
assert abs(Z[0, 1] - np.arctanh(R[0, 1])) < 1e-9, "Z 应等于 arctanh(R)（对角线除外）"
assert abs(r_within - 0.346) < 0.01, "系统内平均 r 应约为 0.35；记得去掉对角线上的 1"
assert abs(r_between - 0.049) < 0.01, "系统间平均 r 应约为 0.05"
print("✓ 任务 1 通过")

## 任务 2：比例阈值——同一矩阵，不同密度，指标如何变

写函数 `proportional_threshold(R, density)`：在上三角中保留 $|r|$ 最大的前 $m=\operatorname{round}(\text{density}\times N(N-1)/2)$ 条边，返回含全部 $N$ 个节点的无向图。

然后在密度 0.05、0.10、0.20、0.30、0.40 上计算平均聚集系数和全局效率，分别存入列表 `C_list`、`E_list`，并把每个密度下的连通分量数存入 `n_comp`。

期望看到：$E$ 随密度单调上升；$C$ 先升后降（系统内的强边用完后，新增的是跨系统的弱边）；低密度时网络断成多个分量。思考：只报告一个密度，会得出什么不同的结论？

> **提示：** `iu = np.triu_indices(N, k=1)`；`np.argsort(w)[::-1][:m]` 取最大的 m 个。

In [ ]:
def proportional_threshold(R, density):
    N = R.shape[0]
    # TODO: 取上三角 |r|，排序后保留前 m 条，建图并返回
    return None


densities = [0.05, 0.10, 0.20, 0.30, 0.40]
C_list, E_list, n_comp = [], [], []
# TODO: 对每个密度建图并计算指标

In [ ]:
# 核对（不用修改）
G_chk = proportional_threshold(R, 0.10)
assert G_chk is not None, "proportional_threshold 还没有返回图"
assert G_chk.number_of_nodes() == 40, "图中应包含全部 40 个节点（包括孤立节点）；先 add_nodes_from(range(N))"
assert G_chk.number_of_edges() == 78, "密度 0.10 应保留 round(0.10×780) = 78 条边"
assert len(E_list) == 5 and all(a < b for a, b in zip(E_list, E_list[1:])), "全局效率应随密度单调上升"
assert abs(C_list[2] - 0.878) < 0.01, "密度 0.20 时 C 应约为 0.88；检查是否按 |r| 从大到小保留"
assert C_list[-1] < C_list[2], "C 应先升后降：密度 0.40 时低于 0.20 时"
assert n_comp[0] > 1 and n_comp[-1] == 1, "低密度时网络应断开，高密度时连通"
print("✓ 任务 2 通过")

## 任务 3：枢纽排序稳定吗？参与系数

已知模块标签 `labels`，写函数 `participation(G, labels)`，返回每个节点的参与系数
$$P_i = 1-\sum_s\left(\frac{k_{is}}{k_i}\right)^2,$$
度为 0 的节点令 $P_i=0$。分别在密度 0.10 和 0.30 下计算 `P10`、`P30`，并给出脑区 9 在两种密度下的名次 `rank9_10`、`rank9_30`（名次 = 参与系数严格大于它的节点数 + 1）。

期望看到：低密度时脑区 9 的跨系统边还没进入网络，$P_9=0$；高密度时它跃居前列——“谁是连接枢纽”依赖阈值。

> **提示：** `np.bincount([labels[j] for j in G.neighbors(i)], minlength=4)` 给出 $k_{is}$。

In [ ]:
# 准备（直接运行）
def proportional_threshold_ref(R, density):
    N = R.shape[0]
    iu = np.triu_indices(N, k=1)
    w = np.abs(R[iu])
    keep = np.argsort(w)[::-1][:int(round(density * len(w)))]
    G = nx.Graph()
    G.add_nodes_from(range(N))
    G.add_edges_from(zip(iu[0][keep], iu[1][keep]))
    return G


R_ref = np.corrcoef(X)

In [ ]:
def participation(G, labels):
    P = np.zeros(G.number_of_nodes())
    # TODO
    return P


P10 = None
P30 = None
rank9_10, rank9_30 = None, None

In [ ]:
# 核对（不用修改）
assert P10 is not None and P30 is not None, "P10、P30 还没有计算"
assert len(P30) == 40 and np.all((P30 >= 0) & (P30 <= 0.75 + 1e-9)), "P 应在 0 到 1 − 1/4 = 0.75 之间"
assert abs(P10[9]) < 1e-9, "密度 0.10 时脑区 9 的边都在系统 0 内，P_9 应为 0"
assert abs(P30[9] - 0.59) < 0.02, "密度 0.30 时 P_9 应约为 0.59"
assert rank9_30 <= 3, "密度 0.30 时脑区 9 的参与系数应排在前 3"
print("✓ 任务 3 通过")

## 任务 4：加权指标与整体缩放

用正相关构造加权网络 $W=\max(R,0)$（对角线为 0）。

1. 写函数 `weighted_efficiency(W)`：把权重转为长度 $\ell_{ij}=1/w_{ij}$，用 Dijkstra 求最短路径长度 $d^w_{ij}$，返回 $E^w=\frac{1}{N(N-1)}\sum_{i\ne j}1/d^w_{ij}$；
2. 计算 `Ew` 和把所有权重乘以 0.7 后的 `Ew_scaled`；
3. 用 `nx.average_clustering(G, weight="weight")`（Onnela 定义）计算 `Cw` 和缩放后的 `Cw_scaled`。

期望看到：`Ew_scaled / Ew` 恰为 0.7，而 `Cw_scaled == Cw`。这就是专题页说的：整体 FC 偏弱的一组，不加处理的加权效率必然偏低，加权聚集系数却不受影响。

> **提示：** `nx.from_numpy_array(W)` 会把矩阵元素存为边属性 `weight`；再给每条边加一个属性 `dist = 1/weight`，然后 `nx.all_pairs_dijkstra_path_length(G, weight="dist")`。

In [ ]:
# 准备（直接运行）
W = np.clip(np.corrcoef(X), 0, None)
np.fill_diagonal(W, 0)

In [ ]:
def weighted_efficiency(W):
    # TODO
    return None


Ew, Ew_scaled = None, None
Cw, Cw_scaled = None, None

In [ ]:
# 核对（不用修改）
assert Ew is not None and Ew_scaled is not None, "Ew、Ew_scaled 还没有计算"
assert abs(Ew - 0.172) < 0.005, "E_w 应约为 0.172；检查是否把权重转成了长度 1/w（强连接 = 短路径）"
assert abs(Ew_scaled / Ew - 0.7) < 1e-6, "所有权重乘以 0.7 后，加权效率也应恰好乘以 0.7"
assert Cw is not None and abs(Cw - Cw_scaled) < 1e-9, "Onnela 加权聚集系数按最大权重归一化，整体缩放不应改变它"
print("✓ 任务 4 通过")

## 任务 5：绝对阈值如何“制造”组间差异

模拟两组各 15 名被试：对照组用 `simulate_bold(seed=s)`，“患者”组把系统信号载荷整体乘以 0.75（`simulate_weak(seed=s)`，网络组织完全相同，只是整体 FC 更弱）。对每名被试：

- 绝对阈值：保留 $r>0.3$ 的边，计算密度和全局效率；
- 比例阈值：密度固定为 0.15（用任务 3 的 `proportional_threshold_ref`），计算全局效率。

把两组的平均值存为 `dens_abs = (对照, 患者)`、`E_abs = (对照, 患者)`、`E_prop = (对照, 患者)`。

期望看到：绝对阈值下患者网络更稀疏、效率更低；比例阈值把密度拉平后，效率差异明显缩小（但不一定为 0——见专题页关于比例阈值会纳入更多噪声边的讨论）。

> **提示：** 二值图可用 `nx.from_numpy_array((R > 0.3) & off)` 构造，注意先去掉对角线。

In [ ]:
# 准备（直接运行）
def simulate_weak(n_mod=4, per=10, T=200, seed=0, scale=0.75):
    """与 simulate_bold 相同，但系统信号的载荷整体乘以 scale（模拟数据）。"""
    rng = np.random.default_rng(seed)
    labels = np.repeat(np.arange(n_mod), per)
    sys_sig = rng.standard_normal((n_mod, T))
    glob = rng.standard_normal(T)
    load = scale * rng.uniform(0.4, 1.0, size=n_mod * per)
    X = load[:, None] * sys_sig[labels] + 0.3 * glob + rng.standard_normal((n_mod * per, T))
    X[9] += scale * 0.8 * sys_sig[1]
    return X, labels


seeds_ctrl = range(100, 115)
seeds_pat = range(200, 215)

In [ ]:
dens_abs = None   # TODO: (对照组平均密度, 患者组平均密度)
E_abs = None      # TODO
E_prop = None     # TODO

In [ ]:
# 核对（不用修改）
assert dens_abs is not None and E_abs is not None and E_prop is not None, "三个结果还没有赋值"
assert dens_abs[1] < dens_abs[0], "整体 FC 更弱时，绝对阈值下患者网络应更稀疏"
assert E_abs[1] < E_abs[0], "绝对阈值下患者组效率应更低"
assert abs(E_prop[0] - E_prop[1]) < abs(E_abs[0] - E_abs[1]), "比例阈值拉平密度后，效率差异应小于绝对阈值下的差异"
print("✓ 任务 5 通过")

## 任务 6：小分析：真实结构连接组的度与功能连接强度

用真实数据 `data/lausanne68_sc_fc.csv`（69 名健康成人、68 个皮层区的结构与功能连接，CC BY 4.0；出处见本讲的真实数据案例和 `data/SOURCES.md`）。setup 已把它读成 `SC`、`FC` 两个 $69\times68\times68$ 数组。

自己决定一个组一致性比例 `consistency`（0.3–0.8 之间），只保留至少这一比例被试中都出现的结构连接，得到二值组网络；计算每个脑区的结构度 `deg_sc`（长度 68）和组平均功能连接的强度 `fc_strength`（先对每人做 Fisher z、平均后反变换，再按行求和，不含对角线）；最后计算二者的 Spearman 相关 `rho`。

换两三个 `consistency` 试一试：结构度高的脑区，功能连接也更强吗？这个关系对阈值敏感吗？可能的混杂是什么（例如脑区大小、追踪偏倚）？

In [ ]:
# 准备（直接运行）
import pandas as pd
from scipy.stats import spearmanr

_df = pd.read_csv("data/lausanne68_sc_fc.csv")
_iu = np.triu_indices(68, 1)


def _mats(mod):
    V = _df[_df.modality == mod].iloc[:, 2:].to_numpy(float)
    M = np.zeros((len(V), 68, 68))
    M[:, _iu[0], _iu[1]] = V
    return M + M.transpose(0, 2, 1)


SC, FC = _mats("SC"), _mats("FC")

In [ ]:
consistency = 0.5
deg_sc = None
fc_strength = None
rho = None

In [ ]:
# 核对（不用修改）
assert 0.3 <= consistency <= 0.8, "consistency 请取 0.3–0.8 之间"
assert deg_sc is not None and np.shape(deg_sc) == (68,), "deg_sc 应是长度 68 的数组"
assert fc_strength is not None and np.shape(fc_strength) == (68,), "fc_strength 应是长度 68 的数组"
assert np.all(np.asarray(deg_sc) >= 0) and np.max(deg_sc) <= 67, "结构度应在 0–67 之间"
assert rho is not None and -1 <= rho <= 1, "rho 应是 −1 到 1 之间的相关系数"
print("✓ 任务 6 通过")